# Data Ingestion - PostgreSQL
Loads normalized CSVs into PostgreSQL sephora_db.
Creates relational constraints (PRIMARY KEY, FOREIGN KEY).
Tables: products, ingredients, product_ingredients, product_skin_types, reviews

In [ ]:
from sqlalchemy import create_engine, text
import pandas as pd
from dotenv import load_dotenv, find_dotenv
import os

In [ ]:
load_dotenv(find_dotenv(), override=True)
engine = create_engine(os.getenv("PG_CONNECTION"))

In [3]:
pd.read_sql("SELECT current_user;", engine)

,current_user
0,atharva


In [ ]:
products_sql = pd.read_csv("../data/processed/products.csv")
ingredients = pd.read_csv("../data/processed/ingredients.csv")
product_ingredients = pd.read_csv("../data/processed/product_ingredients.csv")
reviews_sql = pd.read_csv("../data/processed/reviews.csv", low_memory=False)
product_skin_types = pd.read_csv("../data/processed/product_skin_types.csv")

In [ ]:
products_sql.to_sql("products", engine, if_exists="replace", index=False)
ingredients.to_sql("ingredients", engine, if_exists="replace", index=False)
product_ingredients.to_sql("product_ingredients", engine, if_exists="replace", index=False)
reviews_sql.to_sql("reviews", engine, if_exists="replace", index=False)
product_skin_types.to_sql("product_skin_types", engine, if_exists="replace", index=False, chunksize=10000)

408

In [8]:
from sqlalchemy import text

with engine.connect() as conn:
    conn.execute(text("""
        ALTER TABLE products ADD PRIMARY KEY (product_id);
        ALTER TABLE ingredients ADD PRIMARY KEY (ingredient_id);
        ALTER TABLE product_ingredients 
            ADD FOREIGN KEY (product_id) REFERENCES products(product_id),
            ADD FOREIGN KEY (ingredient_id) REFERENCES ingredients(ingredient_id);
        ALTER TABLE product_skin_types 
            ADD FOREIGN KEY (product_id) REFERENCES products(product_id);
    """))
    conn.commit()

In [ ]:
pd.read_sql("SELECT COUNT(*) FROM products;", engine)
pd.read_sql("SELECT COUNT(*) FROM ingredients;", engine)
pd.read_sql("SELECT COUNT(*) FROM product_ingredients;", engine)
pd.read_sql("SELECT COUNT(*) FROM reviews;", engine)

In [ ]:
conn.execute(text("""
    CREATE OR REPLACE VIEW ingredient_skin_summary AS
    SELECT 
        i.ingredient_name,
        pst.skin_type,
        COUNT(DISTINCT pi.product_id) AS product_count
    FROM ingredients i
    JOIN product_ingredients pi ON i.ingredient_id = pi.ingredient_id
    JOIN product_skin_types pst ON pi.product_id = pst.product_id
    GROUP BY i.ingredient_name, pst.skin_type;
"""))